# 📚 TruthReader Demo — Chatbot Trợ lý Tài liệu Đáng tin cậy

Notebook triển khai hệ thống **TruthReader** — RAG chatbot tài liệu với inline citation, attribution score, và refusal capability.

**Yêu cầu:** Google Colab Pro (A100 40GB) hoặc Kaggle (T4/P100).

| Step | Mô tả |
| :---: | :--- |
| 0 | Kiểm tra GPU |
| 1 | Clone repo & cài dependencies |
| 1b | Restart runtime |
| 2 | Tải models (~32 GB) |
| 3 | Khởi động vLLM server |
| 4 | Chạy Gradio UI |
| 5 | Cleanup |

---

## Step 0: Kiểm tra GPU

Kiểm tra CUDA availability, tên GPU và dung lượng VRAM hiện tại.

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

Wed May 20 12:21:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             50W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Step 1: Clone repository & Cài đặt dependencies

### Cell 1a — Clone repo & thiết lập đường dẫn

Phát hiện môi trường (Colab/Kaggle/Local), clone repo từ GitHub, khởi tạo các biến đường dẫn:
- `REPO_DIR` — thư mục mã nguồn
- `MODEL_DIR` — thư mục lưu models
- `LOG_DIR` — thư mục log
- `TMP_DIR` — thư mục tạm cho file upload

In [ ]:
import os

# Tu dong phat hien moi truong: Colab hay Kaggle
if os.path.exists('/content'):
    ENV = 'colab'
    BASE_DIR = '/content'
elif os.path.exists('/kaggle/working'):
    ENV = 'kaggle'
    BASE_DIR = '/kaggle/working'
else:
    ENV = 'local'
    BASE_DIR = os.path.expanduser('~')

REPO_URL  = 'https://github.com/trantranuit/CS222.git'
REPO_DIR  = os.path.join(BASE_DIR, 'TruthReader-document-assistant')
MODEL_DIR = os.path.join(BASE_DIR, 'models')
LOG_DIR   = os.path.join(BASE_DIR, 'logs')
TMP_DIR   = os.path.join(BASE_DIR, 'tmp_uploads')

print(f'Moi truong: {ENV}')
print(f'BASE_DIR : {BASE_DIR}')
print(f'REPO_DIR : {REPO_DIR}')

if not os.path.exists(REPO_DIR):
    import subprocess
    result = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR],
                           capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError('Clone repo that bai!')
    print('Clone thanh cong!')
else:
    print(f'Repo da ton tai tai {REPO_DIR}')

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

Moi truong: colab
BASE_DIR : /content
REPO_DIR : /content/TruthReader-document-assistant
Repo da ton tai tai /content/TruthReader-document-assistant
Working directory: /content/TruthReader-document-assistant


### Cell 1b — Cài đặt packages

Cài đặt toàn bộ dependencies theo thứ tự:
1. `vllm` — serving LLM qua OpenAI-compatible API
2. Gradio, LangChain, Sentence-Transformers, PDF parsers, PEFT, v.v.
3. Force-reinstall `faiss-cpu` & `onnxruntime-gpu` cho tương thích numpy

> ⚠️ Sau cell này, **bắt buộc Restart Runtime** rồi chạy tiếp từ Step 1b.

In [ ]:

# === BUOC 1: Cai vLLM TRUOC (de no tu keo PyTorch/numpy dung version) ===
!pip install -q vllm 2>&1 | tail -5

# === BUOC 2: Cai cac package con lai (KHONG de torch, numpy, transformers) ===
!pip install -q \
    gradio==4.21.0 \
    langchain \
    langchain-community \
    langchain-core \
    langchain-text-splitters \
    langchain-openai \
    sentence-transformers \
    "openai>=2.0.0" \
    pypdf \
    "pdfminer.six" \
    docx2txt \
    unstructured \
    beautifulsoup4 \
    rouge \
    langdetect \
    PyMuPDF \
    einops \
    jieba \
    nltk \
    "peft>=0.17.0" \
    "optimum[onnxruntime-gpu]" \
    2>&1 | grep -v 'WARNING\|Ignoring invalid\|dependency resolver'

# === BUOC 3: Force-reinstall faiss/onnxruntime cho dung numpy hien tai ===
!pip install -q --force-reinstall --no-deps faiss-cpu onnxruntime-gpu \
    2>&1 | grep -v 'WARNING\|Ignoring invalid'

print('\n✅ Done! Restart Runtime roi chay tu cell tiep theo.')


google-adk 1.29.0 requires websockets<16.0.0,>=15.0.1, but you have websockets 11.0.3 which is incompatible.
cuml-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.2.0 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
pylibraft-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.2.0 which is incompatible.
openai-harmony 0.0.8 requires pydantic>=2.11.7, but you have pydantic 2.9.2 which is incompatible.
vllm 0.21.0 requires pydantic>=2.12.0, but you have pydantic 2.9.2 which is incompatible.
google-genai 1.68.0 requires websockets<17.0,>=13.0.0, but you have websockets 11.0.3 which is incompatible.
cudf-cu12 26.2.1 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.2.0 which is incompatible.
cudf-cu12 26.2.1 requires

## Step 1b: Sau khi Restart Runtime — Chạy từ đây

> **Runtime → Restart session** (Colab) hoặc **Restart Kernel** (Kaggle) rồi chạy cell bên dưới.

Khởi tạo lại biến đường dẫn và import kiểm tra tất cả packages đã cài thành công.

In [ ]:

# === Chay cell nay SAU KHI restart runtime ===
import os, subprocess, sys

# Tu dong phat hien moi truong
if os.path.exists('/content'):
    ENV = 'colab'; BASE_DIR = '/content'
elif os.path.exists('/kaggle/working'):
    ENV = 'kaggle'; BASE_DIR = '/kaggle/working'
else:
    ENV = 'local';  BASE_DIR = os.path.expanduser('~')

REPO_DIR  = os.path.join(BASE_DIR, 'TruthReader-document-assistant')
MODEL_DIR = os.path.join(BASE_DIR, 'models')
LOG_DIR   = os.path.join(BASE_DIR, 'logs')
TMP_DIR   = os.path.join(BASE_DIR, 'tmp_uploads')
os.chdir(REPO_DIR)
print(f'ENV: {ENV}  |  REPO_DIR: {REPO_DIR}')

# Kiem tra nhanh cac package quan trong
import torch, transformers, numpy, peft, langchain, gradio, vllm
print('torch       :', torch.__version__)
print('transformers:', transformers.__version__)
print('numpy       :', numpy.__version__)
print('vllm        :', vllm.__version__)
print('peft        :', peft.__version__)
print('langchain   :', langchain.__version__)
print('gradio      :', gradio.__version__)
print('CUDA        :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU         :', torch.cuda.get_device_name(0))
print('All packages ready!')


ENV: colab  |  REPO_DIR: /content/TruthReader-document-assistant
torch       : 2.11.0+cu130
transformers: 4.57.6
numpy       : 1.26.4
vllm        : 0.21.0
peft        : 0.19.1
langchain   : 1.2.15
gradio      : 4.21.0
CUDA        : True
GPU         : NVIDIA A100-SXM4-80GB
All packages ready!


## Step 2: Tải Models từ HuggingFace

Tải 3 models:
| # | Model | Kích thước |
| :---: | :--- | :---: |
| 1 | `HIT-TMG/bge-m3_RAG-conversational-IR` (Retriever) | ~2.4 GB |
| 2 | `HIT-TMG/Qwen1.5-14B-Chat_RAG-Reader` (Generator) | ~28 GB |
| 3 | `pszemraj/nougat-small-onnx` (OCR) | ~1 GB |

### Cell 2a — Khai báo đường dẫn model

Định nghĩa repo HuggingFace và đường dẫn local cho từng model.

In [ ]:
os.makedirs(MODEL_DIR, exist_ok=True)

EMBED_MODEL     = 'HIT-TMG/bge-m3_RAG-conversational-IR'
GENERATOR_MODEL = 'HIT-TMG/Qwen1.5-14B-Chat_RAG-Reader'  # ~28GB, can A100
OCR_MODEL       = 'pszemraj/nougat-small-onnx'

EMBED_MODEL_PATH     = os.path.join(MODEL_DIR, 'bge-m3_RAG-conversational-IR')
GENERATOR_MODEL_PATH = os.path.join(MODEL_DIR, 'Qwen1.5-14B-Chat_RAG-Reader')
OCR_MODEL_PATH       = os.path.join(MODEL_DIR, 'nougat-small-onnx')

print('MODEL_DIR:', MODEL_DIR)
print('EMBED_MODEL_PATH    :', EMBED_MODEL_PATH)
print('GENERATOR_MODEL_PATH:', GENERATOR_MODEL_PATH)
print('OCR_MODEL_PATH      :', OCR_MODEL_PATH)

MODEL_DIR: /content/models
EMBED_MODEL_PATH    : /content/models/bge-m3_RAG-conversational-IR
GENERATOR_MODEL_PATH: /content/models/Qwen1.5-14B-Chat_RAG-Reader
OCR_MODEL_PATH      : /content/models/nougat-small-onnx


### Cell 2b — Tải Embedding model (Retriever)

Download `HIT-TMG/bge-m3_RAG-conversational-IR` (~2.4 GB) — encode query & documents thành vectors cho FAISS retrieval.

In [ ]:
from huggingface_hub import snapshot_download

# Tải Embedding model (~2.4GB)
if not os.path.exists(EMBED_MODEL_PATH):
    print("⬇️ Đang tải Embedding model...")
    snapshot_download(repo_id=EMBED_MODEL, local_dir=EMBED_MODEL_PATH)
    print("✅ Embedding model đã tải xong!")
else:
    print("✅ Embedding model đã có sẵn.")

✅ Embedding model đã có sẵn.


### Cell 2c — Tải Generator model (Qwen 14B)

Download `HIT-TMG/Qwen1.5-14B-Chat_RAG-Reader` (~28 GB) — sinh câu trả lời kèm inline citation. Sẽ được serve qua vLLM ở Step 3.

In [ ]:
# Tải Generator model (~28GB) - cần thời gian
if not os.path.exists(GENERATOR_MODEL_PATH):
    print("⬇️ Đang tải Generator model (Qwen 14B)... Có thể mất 10-20 phút.")
    snapshot_download(repo_id=GENERATOR_MODEL, local_dir=GENERATOR_MODEL_PATH)
    print("✅ Generator model đã tải xong!")
else:
    print("✅ Generator model đã có sẵn.")

✅ Generator model đã có sẵn.


### Cell 2d — Tải OCR model (Nougat)

Download `pszemraj/nougat-small-onnx` (~1 GB) — chuyển PDF scan thành text có cấu trúc.

In [ ]:
# Tải OCR model (~1GB)
if not os.path.exists(OCR_MODEL_PATH):
    print("⬇️ Đang tải OCR model...")
    snapshot_download(repo_id=OCR_MODEL, local_dir=OCR_MODEL_PATH)
    print("✅ OCR model đã tải xong!")
else:
    print("✅ OCR model đã có sẵn.")

✅ OCR model đã có sẵn.


## Step 3: Khởi động vLLM Server

Chạy Generator model dưới dạng HTTP server (OpenAI-compatible API) tại `http://localhost:8001`.

| Tham số | Giá trị |
| :--- | :--- |
| `--port` | 8001 |
| `--max-model-len` | 5000 |
| `--gpu-memory-utilization` | 0.85 |
| `--dtype` | half (float16) |

### Cell 3a — Khởi động vLLM process

Chạy vLLM server ở background (subprocess). Load model weights vào GPU và lắng nghe requests.

In [ ]:
import subprocess
import time

# Khởi động vLLM server ở background
vllm_process = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", GENERATOR_MODEL_PATH,
        "--served-model-name", "/vllm/EMPTY",
        "--port", "8001",
        "--tensor-parallel-size", "1",
        "--max-model-len", "5000",
        "--gpu-memory-utilization", "0.85",
        "--trust-remote-code",
        "--enforce-eager",
        "--dtype", "half",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("🚀 vLLM server đang khởi động... (PID: {})".format(vllm_process.pid))
print("⏳ Đợi model load xong (có thể mất 2-5 phút)...")

🚀 vLLM server đang khởi động... (PID: 11618)
⏳ Đợi model load xong (có thể mất 2-5 phút)...


### Cell 3b — Đợi vLLM server sẵn sàng

Poll endpoint `/health` mỗi 10s cho đến khi server trả về HTTP 200. Timeout: 600s.

In [ ]:
# Đợi vLLM server sẵn sàng
import urllib.request
import time

def wait_for_vllm(url="http://localhost:8001/health", timeout=600, interval=10):
    """Đợi vLLM server sẵn sàng, tối đa `timeout` giây."""
    start = time.time()
    while time.time() - start < timeout:
        try:
            resp = urllib.request.urlopen(url, timeout=5)
            if resp.status == 200:
                print("\n✅ vLLM server đã sẵn sàng!")
                return True
        except Exception:
            elapsed = int(time.time() - start)
            print(f"\r⏳ Đang đợi vLLM... ({elapsed}s)", end="", flush=True)
            time.sleep(interval)
    print("\n❌ Timeout! vLLM server không khởi động được.")
    return False

vllm_ready = wait_for_vllm()
if not vllm_ready:
    # In log để debug
    print("--- vLLM logs ---")
    for line in vllm_process.stdout:
        print(line, end="")
        if "error" in line.lower():
            break

⏳ Đang đợi vLLM... (60s)
✅ vLLM server đã sẵn sàng!


## Step 4: Chạy TruthReader — Gradio UI

Khởi chạy giao diện web TruthReader. Sau khi chạy xong sẽ tạo public link `https://xxx.gradio.live`.

```
Upload tài liệu → Chunk → FAISS index → User hỏi → Retrieve top-4 → Generate + citation → UI
```

### Cell 4a & 4b — Fix `torchcodec`

Gỡ và cài lại `torchcodec` từ PyTorch wheel index (`cu130`) để tránh xung đột CUDA.

In [ ]:
!pip uninstall -y torchcodec

Found existing installation: torchcodec 0.12.0+cu130
Uninstalling torchcodec-0.12.0+cu130:
  Successfully uninstalled torchcodec-0.12.0+cu130


In [ ]:
!pip install torchcodec --index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://download.pytorch.org/whl/cu130
  Using cached https://download.pytorch.org/whl/cu130/torchcodec-0.12.0%2Bcu130-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (11 kB)
Using cached https://download.pytorch.org/whl/cu130/torchcodec-0.12.0%2Bcu130-cp312-cp312-manylinux_2_28_x86_64.whl (2.7 MB)


### Cell 4c — Tạo thư mục log & tmp

Tạo `LOG_DIR` (lưu conversation log) và `TMP_DIR` (thư mục tạm cho file upload).

In [ ]:
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)
print('LOG_DIR:', LOG_DIR)
print('TMP_DIR:', TMP_DIR)

LOG_DIR: /content/logs
TMP_DIR: /content/tmp_uploads


### Cell 4d — Khởi tạo TruthReader backend

Import modules từ `src/`, gọi `init()` để load Retriever (BGE-M3) vào GPU, kết nối vLLM server, load OCR model.

In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
os.chdir(os.path.join(REPO_DIR, 'src'))

os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'

import gradio as gr
from pathlib import Path
from gradio.events import on
from main import (
    init, example,
    upload_file_fn, clear_file_fn, url_load_fn,
    liked_feedback_collection_fn, chat_listen_fn,
    display_attribution_fn, doc_select_fn, dochelper_chat_fn,
    SPLIT_LINE_PAGE, SPLIT_LINE_DOC,
)

print('Initializing models...')
init(
    embed_model_path=EMBED_MODEL_PATH,
    # Qwen tokenizer path + semicolon-separated jika ada lebih dari 1 model
    chat_model_path=GENERATOR_MODEL_PATH,
    # vLLM server: http://localhost (cung may voi notebook)
    chat_model_server_address='http://localhost',
    ocr_model_path=OCR_MODEL_PATH,
    log_dir=LOG_DIR,
    tmp_folder=TMP_DIR,
)
print('Models loaded!')

Multiple distributions found for package optimum. Picked distribution: optimum
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/content/TruthReader-document-assistant/src/utils.py:185: SyntaxWarning: invalid escape sequence '\d'
  pattern = """<sub><progress class="factual-score progress-\d" value="(.*?)" max="1"></sub>"""
/content/TruthReader-document-assistant/src/utils.py:201: SyntaxWarning: invalid escape sequence '\d'
  pattern = """<a class="citation-button" href="#reference-(\d+)">(\d+)</a>"""
/content/TruthReader-document-assistant/src/main.py:451: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-hu

Initializing models...


/content/TruthReader-document-assistant/src/main.py:451: LangChainDeprecationWarning: `encode_kwargs['show_progress_bar']` was deprecated in LangChain 0.2.5 and will be removed in 1.0. Use the show_progress method on HuggingFaceBgeEmbeddings instead.
  embedding_model = HuggingFaceBgeEmbeddings(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Models loaded!


### Cell 4e — Build Gradio UI & Launch Server

Xây dựng giao diện 2 panel (upload tài liệu + chat), bind event handlers, launch server với `share=True` tạo public link.

In [ ]:
with open(Path(REPO_DIR) / 'src/resources/head.html') as f:
    head = f.read().strip()

with gr.Blocks(
    theme=gr.themes.Soft(font='sans-serif').set(
        background_fill_primary='linear-gradient(90deg, #e3ffe7 0%, #d9e7ff 100%)',
        background_fill_primary_dark='linear-gradient(90deg, #4b6cb7 0%, #182848 100%)',
    ),
    head=head,
    css=Path(REPO_DIR) / 'src/resources/styles.css',
    title='TruthReader',
    fill_height=True,
    analytics_enabled=False,
) as demo:
    doc_text_dict = gr.State()
    doc_pages = gr.State()
    faiss_db = gr.State()
    fake_response_textbox = gr.Textbox(label='fake_response_textbox', visible=False)

    with gr.Row(equal_height=False, elem_classes='row'):
        with gr.Column(min_width=600):
            with gr.Accordion(label='Preprocessing Parameter', open=False) as preprocessing_accordion:
                page_size_slider = gr.components.Slider(minimum=200, maximum=750, step=10, value=750, label='Chunk Size', scale=1)
                page_overlap_slider = gr.components.Slider(minimum=0, maximum=200, step=10, value=0, label='Chunk Overlap', scale=1)
                summary_checkbox = gr.Checkbox(value=True, label='Document Pre-summarization  (cost more time)', scale=1)
                pdf_processor_checkbox = gr.Checkbox(value=False, label='PDF VisionProcessor  (cost more time)', scale=1)

            doc_files_box = gr.File(label='Upload Documents', file_types=['.docx', '.md', '.pdf', '.txt'], file_count='multiple', height=90, scale=1)

            with gr.Group():
                with gr.Row(equal_height=True) as url_row:
                    url_box = gr.Textbox(scale=4, max_lines=1, show_label=False, placeholder="Input URLs (multiple URLs separated by commas ',' )", container=False)
                    url_botton = gr.Button('Fetch Web', size='sm', scale=1)

            with gr.Tabs(visible=False) as tabs:
                with gr.Tab('Document Content', elem_classes='tab', id='tab-doc-cont'):
                    with gr.Accordion(open=True):
                        doc_selection_dropdown = gr.Dropdown(label=None, show_label=False, container=True, interactive=True, scale=1)
                        doc_display_html = gr.HTML(label='Document Content', elem_classes='html-text')
                with gr.Tab('Attribution Chunks', elem_classes='tab', id='tab-attr-chunk'):
                    with gr.Accordion(open=True):
                        attribution_chunks_html = gr.HTML(label='Attribution Chunks', elem_classes='html-text')

        with gr.Column(min_width=600):
            chat_interface = gr.ChatInterface(
                dochelper_chat_fn,
                chatbot=gr.Chatbot(scale=1, label='chatbot', show_label=False, show_copy_button=True, likeable=True, layout='panel', render=False),
                textbox=gr.Textbox(placeholder='Please upload files first.', container=False, scale=7, render=False, max_lines=4, interactive=False, value=None),
                css=Path(REPO_DIR) / 'src/resources/styles.css',
                title='TruthReader',
                theme='soft',
                fill_height=True,
                autofocus=False,
                submit_btn=gr.Button('Submit', variant='primary', scale=1, min_width=150, interactive=False, render=False),
                stop_btn=gr.Button('Stop', variant='stop', visible=False, scale=1, min_width=150, render=False),
                retry_btn=gr.Button('Retry', variant='secondary', interactive=False, render=False, scale=1, size='sm'),
                undo_btn=gr.Button('Undo', variant='secondary', interactive=False, render=False, scale=1, size='sm'),
                clear_btn=gr.Button('Clear', variant='secondary', interactive=False, render=False, scale=1, size='sm'),
                additional_inputs=[
                    gr.Radio(['LyChee-6B', 'Mixtral-7B*2', 'Qwen-14B'], value='Qwen-14B', label='Model Selection', render=True),
                    gr.components.Slider(minimum=0.01, maximum=1.20, step=0.01, value=0.8, label='Temperature', render=False),
                    gr.components.Slider(minimum=0.01, maximum=1.0,  step=0.01, value=0.8, label='Top P', render=False),
                    gr.components.Slider(minimum=1,    maximum=64,   step=1,    value=50,  label='Top K', render=False),
                    gr.components.Slider(minimum=1,    maximum=600,  step=4,    value=600, label='Max New Tokens', render=False),
                    gr.components.Slider(minimum=0,    maximum=1.1,  step=0.01, value=1.02, label='Repetition Penalty', render=False),
                    gr.components.Slider(minimum=0,    maximum=20,   step=1,    value=0,   label='No Repeat Ngram', render=False),
                    gr.Checkbox(value=True, label='Sampling', render=False),
                    faiss_db,
                ],
                additional_inputs_accordion=gr.Accordion(label='Generation Parameter', open=False, render=False),
                concurrency_limit=1,
            )
            gr.Examples(example, inputs=[url_box, chat_interface.textbox])

    # Event bindings
    on([url_box.submit, url_botton.click], url_load_fn,
       [url_box, page_size_slider, page_overlap_slider, summary_checkbox, pdf_processor_checkbox],
       [url_row, preprocessing_accordion, tabs, doc_selection_dropdown, chat_interface.textbox,
        chat_interface.submit_btn, chat_interface.retry_btn, chat_interface.undo_btn,
        chat_interface.clear_btn, doc_text_dict, doc_pages, faiss_db, doc_files_box],
       queue=True,
    ).then(doc_select_fn, [doc_selection_dropdown, doc_text_dict], doc_display_html)

    doc_files_box.upload(upload_file_fn,
        [doc_files_box, page_size_slider, page_overlap_slider, summary_checkbox, pdf_processor_checkbox],
        [url_row, preprocessing_accordion, tabs, doc_selection_dropdown, chat_interface.textbox,
         chat_interface.submit_btn, chat_interface.retry_btn, chat_interface.undo_btn,
         chat_interface.clear_btn, doc_text_dict, doc_pages, faiss_db],
        queue=True, trigger_mode='once',
    ).then(doc_select_fn, [doc_selection_dropdown, doc_text_dict], doc_display_html)

    doc_files_box.clear(clear_file_fn, None,
        [url_row, preprocessing_accordion, tabs, doc_display_html, doc_selection_dropdown,
         chat_interface.textbox, chat_interface.submit_btn, chat_interface.retry_btn,
         chat_interface.undo_btn, chat_interface.clear_btn, chat_interface.chatbot,
         chat_interface.chatbot_state, chat_interface.saved_input, doc_text_dict, doc_pages, faiss_db],
    )

    doc_selection_dropdown.change(doc_select_fn, [doc_selection_dropdown, doc_text_dict], doc_display_html)
    chat_interface.chatbot.like(liked_feedback_collection_fn, chat_interface.chatbot, None)
    chat_interface.chatbot.change(chat_listen_fn, [chat_interface.chatbot_state],
        [fake_response_textbox], queue=False, trigger_mode='always_last')
    fake_response_textbox.change(display_attribution_fn, [fake_response_textbox, doc_text_dict],
        [attribution_chunks_html, tabs], queue=False, trigger_mode='always_last')

print('Launching TruthReader...')
demo.queue(api_open=False).launch(
    server_name='0.0.0.0',
    server_port=8081,
    share=True,   # tao public link, hoat dong tren ca Colab va Kaggle
    show_api=False,
)

Launching TruthReader...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://28fdd9f05681df792c.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


## Step 5: Cleanup

Tắt vLLM server (`SIGTERM`) và giải phóng GPU memory (`torch.cuda.empty_cache()`).

> Sau cleanup, cần chạy lại từ Step 3 nếu muốn dùng tiếp.

In [ ]:
# Dừng vLLM server
try:
    vllm_process.terminate()
    vllm_process.wait(timeout=10)
    print("✅ vLLM server đã dừng.")
except Exception as e:
    print(f"⚠️ Lỗi khi dừng vLLM: {e}")

# Giải phóng GPU memory
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✅ GPU memory đã được giải phóng.")